# מחברת 1: ה-kernel הראשון על A100

המחברת בודקת את הסביבה, מציגה את קוד המקור, בונה ומריצה את `vector_add`. יש להריץ מתוך שורש ה-repo.

In [ ]:
from pathlib import Path
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
print("repo:", ROOT)
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
gpus = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True)
print(gpus.stdout)


אם `nvidia-smi` או `nvcc` מחזירים `None`, עצרו ותקנו את הסביבה. אין טעם לדמות הצלחה של CUDA.

In [ ]:
source = (ROOT / "lessons/01_vector_add.cu").read_text()
print(source)


חפשו בקוד: חישוב אינדקס, בדיקת גבול, שלוש הקצאות device, שתי העתקות H2D, launch, העתקת D2H ואימות התוצאה.

In [ ]:
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "--target", "00_device_query", "01_vector_add", "-j"], check=True)


In [ ]:
device = subprocess.run([str(ROOT / "build/00_device_query")], text=True, capture_output=True, check=True)
print(device.stdout)
assert "A100" in device.stdout, "Selected CUDA device is not identified as an A100; set CUDA_VISIBLE_DEVICES"
assert "compute capability: 8.0" in device.stdout, "Selected device is not compute capability 8.0"


In [ ]:
result = subprocess.run([str(ROOT / "build/01_vector_add")], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0
assert "PASS vector_add" in result.stdout


## שאלות

1. למה צריך `if (i < n)` גם כאשר `n` גדול?
2. מה כולל הזמן שמדפיס ה-CUDA event, ומה הוא אינו כולל?
3. שנו את `n` לערך שאינו מתחלק ב-256 ובדקו שהתוצאה נשארת נכונה.